# Cross-Epoch Discovery: Finding Similar Objects Across All 5 Corpora
**EPS Research RAG Corpus Series**

This notebook demonstrates the unique cross-epoch capability of the EPS platform:
given an object at z=0, find the most similar objects at z~1, z~2, and z~5.

This is the core science use case of the semantic search infrastructure.

In [ ]:
import requests
import pandas as pd

BASE = 'https://dflynn5656-astro-rag-mcp.hf.space/api'

def get_object(corpus, object_id):
    r = requests.get(f'{BASE}/get_object', params={'corpus': corpus, 'object_id': object_id})
    return r.json()['data']['record']

def semantic_search(corpus, query, top_k=5):
    r = requests.get(f'{BASE}/semantic_search', params={'corpus': corpus, 'query': query, 'top_k': top_k})
    return r.json()['data']['results']

print('Ready.')

## Step 1: Choose a z=0 anchor object from the v7 corpus
We start with DDO154 — a well-studied low-mass dwarf irregular in the THINGS survey.

In [ ]:
anchor = get_object('v7', 'DDO154')
print(f"Anchor: {anchor['galaxy']}")
print(f"  Hubble type: {anchor.get('hubble_type','?')}")
print(f"  Distance: {anchor.get('distance_mpc','?')} Mpc")
print(f"  Vrot max: {anchor.get('vrot_max_kms','?')} km/s")
print(f"  Survey: {anchor.get('survey','?')}")

## Step 2: Build a semantic query from the anchor object's properties

In [ ]:
# Build query from anchor properties
query = f"{anchor.get('hubble_type','')} low mass low rotation velocity dwarf irregular"
print(f'Query: "{query}"')

## Step 3: Search across all 5 corpora with the same query

In [ ]:
corpora = ['v7', 'dwarf', 'gc', 'intz', 'z1']
corpus_names = {
    'v7':    'HI Rotation Curves (z=0)',
    'dwarf': 'Dwarf/Irregular (z=0)',
    'gc':    'Globular Clusters (MW)',
    'intz':  'IntZ Kinematics (z~0.4-2.7)',
    'z1':    'ALPINE Z1 (z~4-6)',
}

all_results = {}
for corpus in corpora:
    results = semantic_search(corpus, query, top_k=3)
    all_results[corpus] = results
    print(f"\n{corpus_names[corpus]}:")
    for r in results:
        print(f"  {r['id']:30s}  score={r['score']:.3f}")

## Step 4: Retrieve full records for the top cross-epoch match
Now retrieve the full record for the top intz match to compare with the anchor.

In [ ]:
top_intz_id = all_results['intz'][0]['id']
print(f'Top intz match: {top_intz_id}')

try:
    intz_rec = get_object('intz', top_intz_id)
    ident = intz_rec.get('identifiers', {})
    redsh = intz_rec.get('redshift', {})
    star  = intz_rec.get('stellar_properties', {})
    print(f"  Survey: {ident.get('survey','?')}")
    print(f"  Redshift: {redsh.get('z_spec','?')}")
    print(f"  Log stellar mass: {star.get('log_mass_msun','?')} Msun")
    print(f"  SFR: {star.get('sfr_msun_yr','?')} Msun/yr")
except Exception as e:
    print(f'  (record retrieval: {e})')